In [17]:
import torch

dtype = torch.float
device = torch.device('cpu') # or 'cuda' if you have a GPU

N, D_in, H, D_out = 32, 1000, 100, 10

# 1. The Data (No gradients needed here)
x = torch.randn(N, D_in, device=device, dtype=dtype)
y = torch.randn(N, D_out, device=device, dtype=dtype)

# 2. The Weights (THE BIG CHANGE: Turn the recording switch ON!)
w1 = torch.randn(D_in, H, device=device, dtype=dtype, requires_grad=True)
w2 = torch.randn(H, D_out, device=device, dtype=dtype, requires_grad=True)

learning_rate = 1e-6

for t in range(1000):
    # --- STEP 1: FORWARD PASS ---
    # This is exactly the same as the manual version.
    h = x.mm(w1)
    h_relu = h.clamp(min=0)
    y_pred = h_relu.mm(w2)

    # --- STEP 2: CALCULATE LOSS ---
    # Note: We do NOT use .item() here. We need 'loss' to stay a tracked tensor!
    loss = (y_pred - y).pow(2).sum()
    
    if t % 50 == 49:
        print(t, loss.item())

    # --- STEP 3: BACKPROPAGATION ---
    # We delete all 6 lines of transpose matrix math and replace it with:
    loss.backward()

    # --- STEP 4: UPDATE WEIGHTS ---
    # Turn off the spy cameras (no_grad) so PyTorch doesn't record the update step
    with torch.no_grad():
        # Update the dials using the gradients PyTorch calculated
        w1 -= learning_rate * w1.grad
        w2 -= learning_rate * w2.grad

        # CRITICAL: Manually erase the scratchpad for the next loop!
        w1.grad.zero_()
        w2.grad.zero_()

49 1335.6240234375
99 2.6996917724609375
149 0.00855298899114132
199 0.00010341499000787735
249 1.1993335647275671e-05
299 4.110541340196505e-06
349 2.425077354928362e-06
399 1.7637371456658002e-06
449 1.3524715996027226e-06
499 1.1274959206275526e-06
549 9.541481631458737e-07
599 8.512776048519299e-07
649 7.427314017149911e-07
699 6.42751501800376e-07
749 5.798160032099986e-07
799 5.429355383057555e-07
849 4.901063448414789e-07
899 4.5624148015122046e-07
949 4.2869231720032985e-07
999 4.0349959817831405e-07
